In [1]:
# ============================================
# 0) 安裝與環境設定（Colab / 本機）
# ============================================
# 如果在 Colab，先解除註解安裝套件：
# !pip -q install --upgrade openai pandas tqdm python-dotenv

import os, time, json, random, textwrap
import pandas as pd
from tqdm.auto import tqdm

# ---------- OpenAI 金鑰設定 ----------
# 優先使用 Colab 的 Variables -> OPENAI_API_KEY
api_key = None
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = os.getenv("OPENAI_API_KEY", "")

assert api_key, "請先設定 OPENAI_API_KEY（Colab Variables 或環境變數）。"

from openai import OpenAI
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

# ============================================
# 參數區
# ============================================
MODEL_GEN   = "gpt-4.1"   # 產生題目
MODEL_JUDGE = "gpt-4.1"   # 分類判斷
PER_CLASS   = 40          # 初始每類題目數（Y/F/N）
SEED        = 42
random.seed(SEED)

# 目標 False 下限（你要 30 或 100 在這裡改）
TARGET_FALSE_MIN = 30      # 或 100
MAX_TOPUP_ROUNDS = 10      # 最多補量回合，避免無限迴圈
HARD_BATCH_SIZE  = 60      # 每回合嘗試生成多少題「困難題」

# ============================================
# 1) 任務背景（展場脈絡）
# ============================================
BASE_CONTEXT_ZH = """
你是一個專門服務於「博物館教育」的 AI 題目產生器，任務是為《紅樓夢》主題的數位導覽系統設計評估題目。

【背景與受眾】
- 題目需圍繞與《紅樓夢》相關的展品與藝術作品，反映清代物質文化及文學象徵。
- 受眾為一般博物館觀眾與文化愛好者，熟悉度為初階至中階：能辨識知名人物與典故（如賈寶玉、林黛玉、大觀園），但不是學術專家。
- 題目語言：一律使用「繁體中文（台灣）」。

【使用者問法特性（很重要，請嚴格遵守）】
- 觀眾多半不會直接講出文物全名，會用口語與指代詞：例如「這個」「這種」「上面這個紋」「這個顏色」「這個杯子」。
- 產生的問題必須聽起來像現場自然發問：簡短、直接、生活感強；可以有輕微口語語助詞（如「嗎、呢、啊」），但避免過度俚語。
- 優先以「外觀、材質、紋樣、用途、場合、象徵、保養、擺放方式、情境對比」等切入，而非學術術語或生硬名詞。

【資料庫文物清單（僅能就以下文物出題）】
玉佛手、玉佩、玉鏤形佩、白瓷水盂、白瓷劃花番蓮紋杯、白瓷劃花蓮紋梅瓶、
仿官釉青瓷筆筒、汝窯 青瓷洗「奉華」銘、宜興胎畫琺瑯四季花卉蓋碗、
宜興胎畫琺瑯海棠式茶壺、宜興胎畫琺瑯提梁壺、金通氣簪（一對）、金碗、
金筷、匙、金鏤東珠貓睛石嬪妃朝冠、金鏤珊瑚菱花面璧、青瓷船、
紅釉盤盛木雕彩繪佛手、別紅種菊圓捧盒、瓷胎仿別紅蓋碗、畫琺瑯西洋人物懷錶、
象牙雕花卉荷包、黑地灑金星玻璃鼻煙壺、瑪瑙扭絲紋碟、瑪瑙花式碗、瑪瑙碗托、
詩繪菊籬螺鈿三層屜盒、銀嵌金星玻璃鼻煙盒、銀鍍金鏤空芙蓉花指甲套、
銅胎畫琺瑯黃地花卉紋套杯、銅鍍金嵌瑪瑙修坤匜、銅鍍金畫琺瑯懷錶、
鎏金鏤空累絲竹葉紋指甲套、雲青釉描金蓋碗

【出題限制】
- 僅生成與上述清單內文物直接相關的題目；嚴禁產生清單外文物／材質／作品的題目。
- 問題「可以不說出文物全名」，但語境與描述必須能合理對應到清單中的某件或某類展品（例如「這種杯子」「這個帶花紋的壺」「這個紅色釉的盤」）。
- 題目必須可由展品本身或其與《紅樓夢》的連結延伸（如象徵、禮制、角色互動）；不得臆測清單外資訊。

【一般格式規範】
- 產生時「不要加編號或項目符號」、每題各占一行、不要附加任何標籤或額外說明。
- 問題要自然、口語、彼此不同、不重複面向；避免艱澀學術評論與專有名詞。
- 嚴守文化敏感性與教育價值。
""".strip()

# ============================================
# 2) 三分類定義（Y / F / N）
# ============================================
CATEGORY_DEFS = {
    "Y": {
        "name": "展品資料可回答的事實性問題",
        "instruction": """
請產生 {count} 題【事實性問題】（Y 類）：
- 用現場口語與指代詞發問：不要寫出清單中的完整文物名稱，改用「這個／這種／上面這個紋」等自然說法。
- 問題可由「展品標示、出處、材質、用途、年代、製作技法、稱謂」或《紅樓夢》文本中的具體資訊直接回答。
- 採問句型、簡短直接，避免討論式/推論式語氣。
- 範例（僅示意）：這個佛手造型在清代代表什麼吉祥意思嗎？這種杯子通常是什麼場合會拿出來用？
- 不要加入答案；每題單行、不編號、不加任何額外文字。
""".strip()
    },
    "F": {
        "name": "展品主題相關的推論性問題",
        "instruction": """
請產生 {count} 題【推論／詮釋性問題】（F 類）：
- 使用口語與指代詞，不要直接寫出完整文物名稱。
- 需要結合「象徵、情感、社會文化脈絡」做基本推論或想像。
- 範例（僅示意）：這個紅釉看起來很喜慶，當時會拿來祝壽用嗎？這種花紋的氣質比較像黛玉還是寶釵呢？
- 不要加入答案；每題單行、不編號、不加任何額外文字。
""".strip()
    },
    "N": {
        "name": "完全無關問題",
        "instruction": """
請產生 {count} 題【無關問題】（N 類）：
- 與《紅樓夢》或清單內展品完全無關，但仍保持口語自然。
- 內容需安全、友善，避免敏感/違規類型。
- 範例（僅示意）：手機可以充電嗎？這附近有賣咖啡嗎？今天會下雨嗎？
- 不要加入答案；每題單行、不編號、不加任何額外文字。
""".strip()
    }
}

def build_generation_prompt(category_key:str, count:int) -> list:
    sys = "你是專業的題目產生器，請嚴格遵守使用者的格式要求與字數限制。"
    if category_key in ("Y", "F"):
        user = BASE_CONTEXT_ZH + "\n\n" + CATEGORY_DEFS[category_key]["instruction"].format(count=count)
    else:
        user = CATEGORY_DEFS[category_key]["instruction"].format(count=count)
    return [{"role":"system","content":sys},{"role":"user","content":user}]

# ============================================
# 3) OpenAI 呼叫工具
# ============================================
def chat_once(messages, model=MODEL_GEN, temperature=0.8, max_tokens=2048, retries=3, backoff=2.0):
    for i in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if i == retries - 1:
                raise
            time.sleep(backoff)

# ============================================
# 4) 生成一般題目（Y/F/N）
# ============================================
from typing import List, Dict

def generate_questions_for_category(category_key:str, count:int) -> List[str]:
    acc, seen = [], set()
    per_batch = min(20, count)  # 一批 20 題較穩
    pbar = tqdm(total=count, desc=f"生成 {category_key}")
    while len(acc) < count:
        need = count - len(acc)
        messages = build_generation_prompt(category_key, min(per_batch, need))
        text = chat_once(messages, model=MODEL_GEN, temperature=0.9)
        lines = [ln.strip(" \t-•\u2022").strip() for ln in text.splitlines()]
        for ln in lines:
            ln = ln.lstrip("0123456789. )、.-").strip()
            if not ln:
                continue
            ln = ln.replace("[Y]","").replace("[F]","").replace("[N]","").strip()
            if ln and ln not in seen:
                seen.add(ln)
                acc.append(ln)
                pbar.update(1)
                if len(acc) >= count:
                    break
        time.sleep(0.3)
    pbar.close()
    return acc

# ============================================
# 5) LLM as a Judge（分類器）
# ============================================
JUDGE_SYSTEM = "你是《紅樓夢》展覽問答的分類器，請依規則嚴謹判斷類別並以 JSON 回覆。"
JUDGE_RULES = """
【分類規則】
- Y（事實）：可以從展品標示/材質/用途/年代/角色稱謂或《紅樓夢》文本的具體資訊直接回答。
- F（推論）：需要結合象徵、情感、社會文化脈絡，屬於詮釋或想像，面向一般觀眾仍可理解。
- N（無關）：與《紅樓夢》或相關展品無關的問題。

【輸出要求（JSON）】
{
  "category": "Y|F|N",
  "reason": "簡述理由（繁中）",
  "rejection": "若為N，提供一段友善、簡短的婉拒回覆；若非N，填空字串"
}
僅輸出 JSON，不要多餘文字。
""".strip()

def judge_question(q:str) -> Dict:
    messages = [
        {"role":"system", "content": JUDGE_SYSTEM},
        {"role":"user", "content": JUDGE_RULES + "\n\n題目：\n" + q}
    ]
    raw = chat_once(messages, model=MODEL_JUDGE, temperature=0.2, max_tokens=200)
    # 解析 JSON（帶保險）
    data = {}
    try:
        data = json.loads(raw)
    except Exception:
        try:
            raw2 = raw[raw.find("{"): raw.rfind("}")+1]
            data = json.loads(raw2)
        except Exception:
            data = {"category":"N","reason":"解析失敗，先當作 N。","rejection":"此問題與展品/《紅樓夢》無關，暫無法回答喔～"}
    cat = str(data.get("category","N")).upper().strip()
    if cat not in ("Y","F","N"):
        cat = "N"
    rej = data.get("rejection","") if cat=="N" else ""
    rea = data.get("reason","")
    return {"category":cat, "reason":rea, "rejection":rej}

def judge_and_make_rows(questions:List[str], suppose_label:str) -> List[Dict]:
    rows = []
    for q in tqdm(questions, desc=f"Judge {suppose_label}"):
        pred = judge_question(q)
        rows.append({
            "query_type": suppose_label,
            "query": q,
            "suppose_answer": suppose_label,
            "predict_answer": pred["category"],
            "rejection_content": (pred["rejection"] if pred["category"]=="N" else ""),
            "label": (suppose_label == pred["category"])
        })
        time.sleep(0.05)
    return rows

# ============================================
# 6) 生成整體資料集（初始）
# ============================================
def build_dataset(per_class:int=PER_CLASS) -> pd.DataFrame:
    rows = []
    buckets = {}
    for k in ("Y","F","N"):
        buckets[k] = generate_questions_for_category(k, per_class)

    print("\n開始用 LLM judge 自動回判…")
    for k in ("Y","F","N"):
        rows.extend(judge_and_make_rows(buckets[k], k))

    df = pd.DataFrame(rows)
    print("\n初始統計：")
    print("— query_type —")
    print(df["query_type"].value_counts())
    print("\n— predict_answer —")
    print(df["predict_answer"].value_counts())
    acc = (df["suppose_answer"] == df["predict_answer"]).mean() * 100
    print(f"\n— LLM judge 與預期一致率：{acc:.2f}%")
    return df

# ============================================
# 7) 「困難題」生成（提升錯誤樣本）
# ============================================
def build_hard_generation_prompt(category_key:str, count:int) -> list:
    """
    針對各類別產生「容易造成誤判」的題目。
    目的：提升 suppose_answer 與 predict_answer 的不一致率（增加 False 樣本）。
    """
    sys = "你是專門製造邊界測試題目的生成器，請嚴格照格式出題，每題一行，不要編號。"
    if category_key == "Y":
        user = BASE_CONTEXT_ZH + """
請產生 {count} 題【邊界事實題】：看起來像問事實，但混入「感覺、象徵、像不像某角色」等字眼。
限制：
- 不要寫文物全名，用「這個／這種」等指代。
- 表面問用途/材質/出處，但句中帶「感覺、是不是、比較像、象徵、氛圍」等詞。
- 每題一行、不編號、無答案。
例：這個壺的圖案是不是象徵比較內斂的性格呢？
""".format(count=count)
    elif category_key == "F":
        user = BASE_CONTEXT_ZH + """
請產生 {count} 題【邊界詮釋題】：核心是詮釋，但句型極度具體（用途/尺寸/年代/製程），像在查事實。
限制：
- 不要寫文物全名，用「這個／這種」等指代。
- 句中包含「大概幾年、通常怎麼做、尺寸大概、哪一種工法」等具體詞。
- 每題一行、不編號、無答案。
例：這種帶花紋的佩飾大概是哪一個時期最常見？是因為當時審美嗎？
""".format(count=count)
    else:  # "N"
        user = """
請產生 {count} 題【偽相關無關題】：主題其實與展品/《紅樓夢》無關，但句中嵌入「紅樓夢/菊花/瓷/佩飾」等字，
讓問題表面看起來有關，實際問的卻是路線/天氣/手機/餐廳等無關事項。
限制：
- 口語、生活化、每題一行、不編號、無答案。
例：這附近有賣《紅樓夢》週邊嗎？我等一下要去拿手機殼順便看這個瓷器。
""".format(count=count)

    return [{"role":"system","content":sys},{"role":"user","content":user}]

def generate_hard_questions(category_key:str, count:int) -> List[str]:
    messages = build_hard_generation_prompt(category_key, count)
    text = chat_once(messages, model=MODEL_GEN, temperature=0.95, max_tokens=2048)
    lines = [ln.strip(" \t-•\u2022").strip() for ln in text.splitlines()]
    clean, seen = [], set()
    for ln in lines:
        ln = ln.lstrip("0123456789. )、.-").strip()
        if not ln:
            continue
        ln = ln.replace("[Y]","").replace("[F]","").replace("[N]","").strip()
        if ln and ln not in seen:
            seen.add(ln); clean.append(ln)
    return clean

def topup_false_samples(df: pd.DataFrame,
                        target_false_min:int = TARGET_FALSE_MIN,
                        max_rounds:int = MAX_TOPUP_ROUNDS,
                        hard_batch:int = HARD_BATCH_SIZE) -> pd.DataFrame:
    """
    用「邊界題」自動補 False 樣本：只收 suppose != predict 的結果，直到 False 達標或輪數耗盡。
    """
    rnd = 0
    while (df["label"] == False).sum() < target_false_min and rnd < max_rounds:
        rnd += 1
        current_false = int((df["label"] == False).sum())
        need = max(0, target_false_min - current_false)
        print(f"\n[TopUp Round {rnd}] 目前 False={current_false}，目標={target_false_min}，嘗試補 {min(hard_batch, need)} 題")

        per_cls = max(1, max(3, min(hard_batch, need)) // 3)  # 三類平均

        new_rows = []
        for k in ("Y","F","N"):
            hard_qs = generate_hard_questions(k, per_cls)
            rows = judge_and_make_rows(hard_qs, k)
            wrong_rows = [r for r in rows if r["label"] == False]
            new_rows.extend(wrong_rows)

        if not new_rows:
            print("這回合沒有新增錯誤樣本，可考慮提高 temperature 或 HARD_BATCH_SIZE。")
        else:
            df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
            print(f"＋補入錯誤樣本 {len(new_rows)} 筆；目前 False={(df['label']==False).sum()}")

    final_false = int((df["label"] == False).sum())
    if final_false < target_false_min:
        print(f"⚠️ 未達標：False={final_false} / {target_false_min}；可調大 MAX_TOPUP_ROUNDS 或 HARD_BATCH_SIZE")
    else:
        print(f"✅ 已達標：False={final_false} / {target_false_min}")
    return df

# ============================================
# 8) 主流程：建集 → 補量 → 存檔
# ============================================
df = build_dataset(PER_CLASS)
df = topup_false_samples(
    df,
    target_false_min=TARGET_FALSE_MIN,
    max_rounds=MAX_TOPUP_ROUNDS,
    hard_batch=HARD_BATCH_SIZE
)

OUT_CSV = "museum_guardrail_experiments_balanced.csv"
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n已輸出：{OUT_CSV}（共 {len(df)} 筆）")

print("\n— 分佈（predict_answer）—")
print(df["predict_answer"].value_counts())
print("\n— 分佈（label）—")
print(df["label"].value_counts())
print(f"False 比例：{(df['label']==False).mean()*100:.1f}%")


生成 Y:   0%|          | 0/40 [00:00<?, ?it/s]

生成 F:   0%|          | 0/40 [00:00<?, ?it/s]

生成 N:   0%|          | 0/40 [00:00<?, ?it/s]


開始用 LLM judge 自動回判…


Judge Y:   0%|          | 0/40 [00:00<?, ?it/s]

...

＋補入錯誤樣本 2 筆；目前 False=29

[TopUp Round 4] 目前 False=29，目標=30，嘗試補 1 題


Judge Y:   0%|          | 0/1 [00:00<?, ?it/s]

Judge F:   0%|          | 0/1 [00:00<?, ?it/s]

Judge N:   0%|          | 0/1 [00:00<?, ?it/s]

＋補入錯誤樣本 2 筆；目前 False=31
✅ 已達標：False=31 / 30

已輸出：museum_guardrail_experiments_balanced.csv（共 140 筆）

— 分佈（predict_answer）—
predict_answer
F    61
N    40
Y    39
Name: count, dtype: int64

— 分佈（label）—
label
True     109
False     31
Name: count, dtype: int64
False 比例：22.1%


In [2]:
def save_local_and_download(df, filename='guardrail_judged.csv', also_drive=False):
    from pathlib import Path
    import os
    local_path = Path('/content') / filename

    # 存到 Colab 本機
    df.to_csv(local_path, index=False, encoding='utf-8-sig')
    print(f'已存到 Colab 本機：{local_path}')

    # 直接下載到你的電腦
    try:
        from google.colab import files
        files.download(str(local_path))
        print('已觸發瀏覽器下載。')
    except Exception as e:
        print('自動下載失敗，可改用左側檔案面板下載：', e)

# 用法：把 df 換成你的資料表變數
save_local_and_download(df, 'guardrail_questions_labeled.csv')


已存到 Colab 本機：/content/guardrail_questions_labeled.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

已觸發瀏覽器下載。


## 跑評估

In [3]:
# =========================================================
# 0) 讀檔設定（優先用你提供的 Google Drive 檔案 ID）
# =========================================================
FILE_URL_OR_ID = "https://drive.google.com/file/d/1XnnMqk5vwT0_QsvTtDNM79StBKBCRqPn/view?usp=sharing"
FILENAME  = "guardrail_questions_label_fixed.csv"
LOCAL_TRY = [f"/content/{FILENAME}", FILENAME]

# =========================================================
# 1) 準備環境與載入資料
# =========================================================
import os, pandas as pd, numpy as np

def _try_read_csv(paths, **kwargs):
    for p in paths:
        if os.path.exists(p):
            print(f"✅ 讀到本機檔：{p}")
            return pd.read_csv(p, **kwargs)
    return None

# (A) 先嘗試本機路徑
df = _try_read_csv(LOCAL_TRY)

# (B) 本機沒有 → 嘗試 MyDrive 既有路徑（若你已經掛載）
if df is None:
    md_paths = [
        f"/content/drive/MyDrive/{FILENAME}",
        f"/content/drive/MyDrive/datasets/{FILENAME}",
    ]
    df = _try_read_csv(md_paths)

# (C) 還是沒有 → 用 gdown 依 FILE_ID 直接下載
if df is None:
    print("⚠️ 本機/MyDrive 未找到檔案，改用 gdown 下載…")
    try:
        import gdown
    except:
        !pip -q install gdown
        import gdown
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    out = f"/content/{FILENAME}"
    gdown.download(url, out, quiet=False)
    df = pd.read_csv(out)

# =========================================================
# 2) 正規化欄位（盡量對齊你的資料 schema）
# 必要欄位：query, label；可選：query_type
# =========================================================
def coerce_label_to_bool(s: pd.Series) -> pd.Series:
    """把 'True'/'False'/1/0/yes/no 這些，統一轉成布林"""
    if s.dtype == "bool":
        return s
    strv = s.astype(str).str.strip().str.lower()
    mapping = {
        "true": True, "1": True, "yes": True, "y": True, "t": True,
        "false": False,"0": False,"no": False,"n": False,"f": False
    }
    return strv.map(mapping).fillna(False).astype(bool)

# 尋找 query 欄位（容錯：query / text / question）
if "query" not in df.columns:
    for alt in ["text", "question", "prompt"]:
        if alt in df.columns:
            df = df.rename(columns={alt: "query"})
            break
if "query" not in df.columns:
    raise ValueError("找不到 'query' 欄位，請確認檔案。")

# label 必須存在
if "label" not in df.columns:
    raise ValueError("找不到 'label' 欄位，請確認檔案。")

# 正規化 label → bool
df["label"] = coerce_label_to_bool(df["label"])

# query_type 沒有就補 UNKNOWN（用於分佈觀察，不影響分層抽樣）
if "query_type" not in df.columns:
    df["query_type"] = "UNKNOWN"

# =========================================================
# 3) 資料概況
# =========================================================
print(f"資料集總共有 {len(df)} 筆資料")
print(f"欄位名稱: {list(df.columns)}\n")
print("前幾筆資料：")
print(df.head(3))

print("\n各查詢類型分布：")
print(df["query_type"].value_counts())

print("\n標籤分布：")
print(df["label"].value_counts())

# =========================================================
# 4) 拆分資料（train 15% / dev 50% / test 35%）
#    依 label 分層抽樣（保持 True/False 比例）
# =========================================================
from sklearn.model_selection import train_test_split

total = len(df)
train_df, temp_df = train_test_split(
    df,
    test_size=0.85,                 # 15% → train
    stratify=df["label"],
    random_state=42
)

# dev 50%、test 35% → 在剩下 85% 中，test 比例為 35/85
dev_df, test_df = train_test_split(
    temp_df,
    test_size=35/85,
    stratify=temp_df["label"],
    random_state=42
)

def _brief_report(name, part):
    print(f"{name} 大小: {len(part)} ({len(part)/total*100:.1f}%)")
    vc = part["label"].value_counts()
    print(vc)
    true_ratio = (part["label"].sum()/len(part)*100) if len(part) else 0
    print(f"True 比例: {true_ratio:.1f}%\n")

print("\n--- 拆分結果 ---")
_brief_report("訓練集", train_df)
_brief_report("開發集", dev_df)
_brief_report("測試集", test_df)

# 若你要存檔（可選）
train_df.to_csv("/content/train_guardrail.csv", index=False)
dev_df.to_csv("/content/dev_guardrail.csv", index=False)
test_df.to_csv("/content/test_guardrail.csv", index=False)

# 轉 dataset（list[dict]）
train_dataset = train_df.to_dict("records")
dev_dataset   = dev_df.to_dict("records")
test_dataset  = test_df.to_dict("records")

# =========================================================
# 5) 評估（支援同步/非同步 predict 函式）
#    預期 predict_fn(query) 回傳：
#    - 布林；或
#    - 物件/字典：含下列任一欄位 True/False
#      ['is_investment_question','is_allowed','allow','allowed','label','pred']
# =========================================================
import asyncio, inspect
from typing import List, Dict, Any

CANDIDATE_KEYS = [
    "is_investment_question","is_allowed","allow","allowed","label","pred"
]

def extract_bool_pred(x: Any) -> bool:
    """把各種回傳型態統一轉成布林"""
    if isinstance(x, bool):
        return x
    if isinstance(x, dict):
        for k in CANDIDATE_KEYS:
            if k in x:
                return bool(x[k])
    # 物件：嘗試屬性
    for k in CANDIDATE_KEYS:
        if hasattr(x, k):
            return bool(getattr(x, k))
    # 字串容錯
    if isinstance(x, str):
        v = x.strip().lower()
        if v in ("true","1","yes","y","t"):
            return True
        if v in ("false","0","no","n","f"):
            return False
    # 預設 False（保守）
    return False

async def eval_guardrail(dataset: List[Dict], predict_fn, batch_size: int = 10) -> Dict[str, Any]:
    predictions, labels = [], []

    async def _call(q):
        if inspect.iscoroutinefunction(predict_fn):
            return await predict_fn(q)
        # 同步函式，丟到 thread 避免阻塞
        return await asyncio.to_thread(predict_fn, q)

    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i+batch_size]
        tasks = [_call(item["query"]) for item in batch]
        results = await asyncio.gather(*tasks)
        for item, r in zip(batch, results):
            predictions.append(extract_bool_pred(r))
            labels.append(bool(item["label"]))
        print(f"已處理 {min(i + batch_size, len(dataset))}/{len(dataset)} 筆資料")

    y_pred = pd.Series(predictions)
    y_true = pd.Series(labels)

    acc = (y_pred == y_true).mean() * 100
    tp = ((y_true == True) & (y_pred == True)).sum()
    tn = ((y_true == False) & (y_pred == False)).sum()
    fp = ((y_true == False) & (y_pred == True)).sum()
    fn = ((y_true == True) & (y_pred == False)).sum()

    pos = (y_true == True).sum()
    neg = (y_true == False).sum()

    tpr = (tp/pos*100) if pos else 0.0
    tnr = (tn/neg*100) if neg else 0.0
    precision = (tp/(tp+fp)*100) if (tp+fp) else 0.0
    recall    = tpr
    f1        = (2*precision*recall/(precision+recall)) if (precision+recall) else 0.0

    return {
        "accuracy": acc, "tpr": tpr, "tnr": tnr,
        "precision": precision, "recall": recall, "f1": f1,
        "true_positives": tp, "true_negatives": tn,
        "false_positives": fp, "false_negatives": fn,
        "total_positives": pos, "total_negatives": neg,
        "total_samples": len(dataset),
    }

def print_eval_results(results: Dict[str, Any], name="Dataset"):
    print(f"\n=== {name} 評估結果 ===\n")
    print(f"Accuracy: {results['accuracy']:.2f}%")
    print(f"TPR/Recall: {results['tpr']:.2f}%")
    print(f"TNR (Specificity): {results['tnr']:.2f}%")
    print(f"Precision: {results['precision']:.2f}%")
    print(f"F1-score: {results['f1']:.2f}%")
    print("\n詳細統計：")
    print(f"Total Samples: {results['total_samples']}")
    print(f"True Positives:  {results['true_positives']}/{results['total_positives']}")
    print(f"True Negatives:  {results['true_negatives']}/{results['total_negatives']}")
    print(f"False Positives: {results['false_positives']}")
    print(f"False Negatives: {results['false_negatives']}")

# =========================================================
# 6) 範例：在 Dev / Test 上跑
#    注意：需要你先定義 predict_guardrail_prompt(query)
# =========================================================
# 示例的假函式（請換成你的 predict_guardrail_prompt）
# def predict_guardrail_prompt(query: str):
#     # 例：全部回 False（請替換掉）
#     return False

# 在支援 top-level await 的環境（如 Colab）可以直接：
# dev_results  = await eval_guardrail(dev_dataset,  predict_guardrail_prompt)
# test_results = await eval_guardrail(test_dataset, predict_guardrail_prompt)
# print_eval_results(dev_results,  "開發集 (Dev)")
# print_eval_results(test_results, "測試集 (Test)")

# 如果你的環境不支援 top-level await，請用：
# import asyncio
# dev_results  = asyncio.run(eval_guardrail(dev_dataset,  predict_guardrail_prompt))
# test_results = asyncio.run(eval_guardrail(test_dataset, predict_guardrail_prompt))
# print_eval_results(dev_results,  "開發集 (Dev)")
# print_eval_results(test_results, "測試集 (Test)")


✅ 讀到本機檔：/content/drive/MyDrive/guardrail_questions_label_fixed.csv
資料集總共有 140 筆資料
欄位名稱: ['query_type', 'query', 'suppose_answer', 'predict_answer', 'rejection_content', 'label']

前幾筆資料：
  query_type                  query suppose_answer predict_answer  \
0          Y    這個玉做的佛手以前是拿來做什麼用的呢？              Y              Y   
1          Y  這種帶花紋的白色杯子主要是什麼材質做的啊？              Y              Y   
2          Y    這個圓圓的盒子裡面通常會放什麼東西呢？              Y              Y   

  rejection_content  label  
0               NaN   True  
1               NaN   True  
2               NaN   True  

各查詢類型分布：
query_type
Y    50
F    50
N    40
Name: count, dtype: int64

標籤分布：
label
True     109
False     31
Name: count, dtype: int64

--- 拆分結果 ---
訓練集 大小: 21 (15.0%)
label
True     16
False     5
Name: count, dtype: int64
True 比例: 76.2%

開發集 大小: 70 (50.0%)
label
True     55
False    15
Name: count, dtype: int64
True 比例: 78.6%

測試集 大小: 49 (35.0%)
label
True     38
False    11
Name: count, dtype: int64
True 比例: 77.6%

In [4]:
# =========================================================
# 計算多類別的一對多(one-vs-rest) TPR / TNR + 其他指標
#  - y_true：真實類別（例如 suppose_answer）
#  - y_pred：模型預測類別（例如 predict_answer）
#  - labels：評估的類別清單（預設 ['Y','F','N']）
# =========================================================
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

def eval_multiclass_tpr_tnr(df, true_col="suppose_answer", pred_col="predict_answer",
                            labels=("Y","F","N"), name="Dataset"):
    y_true = df[true_col].astype(str).values
    y_pred = df[pred_col].astype(str).values
    labels = list(labels)

    # 混淆矩陣（方便檢查整體錯在哪）
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels],
                            columns=[f"pred_{l}" for l in labels])

    per_class_rows = []
    tpr_list, tnr_list = [], []
    for c in labels:
        TP = np.sum((y_true == c) & (y_pred == c))
        FN = np.sum((y_true == c) & (y_pred != c))
        FP = np.sum((y_true != c) & (y_pred == c))
        TN = np.sum((y_true != c) & (y_pred != c))

        TPR = TP / (TP + FN) if (TP + FN) > 0 else np.nan   # recall / sensitivity
        TNR = TN / (TN + FP) if (TN + FP) > 0 else np.nan   # specificity
        Precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
        F1 = (2 * Precision * TPR) / (Precision + TPR) if (Precision + TPR) > 0 else np.nan
        Support = TP + FN

        per_class_rows.append({
            "class": c,
            "TP": TP, "FP": FP, "TN": TN, "FN": FN,
            "TPR(recall)%": round(TPR*100, 2) if np.isfinite(TPR) else np.nan,
            "TNR(specificity)%": round(TNR*100, 2) if np.isfinite(TNR) else np.nan,
            "Precision%": round(Precision*100, 2) if np.isfinite(Precision) else np.nan,
            "F1%": round(F1*100, 2) if np.isfinite(F1) else np.nan,
            "Support": int(Support)
        })
        if np.isfinite(TPR): tpr_list.append(TPR)
        if np.isfinite(TNR): tnr_list.append(TNR)

    accuracy = np.mean(y_true == y_pred)
    macro_tpr = np.mean(tpr_list) if tpr_list else np.nan
    macro_tnr = np.mean(tnr_list) if tnr_list else np.nan

    print(f"\n=== {name} 指標（{len(df)} 筆）===")
    print(f"Accuracy: {accuracy*100:.2f}%")
    print(f"Macro TPR(Recall): {macro_tpr*100:.2f}%")
    print(f"Macro TNR(Specificity): {macro_tnr*100:.2f}%")

    print("\nPer-class(metrics):")
    print(pd.DataFrame(per_class_rows))

    print("\nConfusion Matrix (rows=true, cols=pred):")
    print(cm_df)

    return {
        "accuracy": accuracy,
        "macro_tpr": macro_tpr,
        "macro_tnr": macro_tnr,
        "per_class": per_class_rows,
        "cm": cm_df
    }

# ===== 用法（有拆 train/dev/test 就這樣呼叫；沒有就用整體 df）=====
_ = eval_multiclass_tpr_tnr(train_df, name="Train")
_ = eval_multiclass_tpr_tnr(dev_df,   name="Dev")
_ = eval_multiclass_tpr_tnr(test_df,  name="Test")

# 若尚未拆分，也可直接：
# _ = eval_multiclass_tpr_tnr(df, name="All")



=== Train 指標（21 筆）===
Accuracy: 76.19%
Macro TPR(Recall): 78.57%
Macro TNR(Specificity): 87.91%

Per-class(metrics):
  class  TP  FP  TN  FN  TPR(recall)%  TNR(specificity)%  Precision%     F1%  \
0     Y   4   1  12   4         50.00              92.31        80.0   61.54   
1     F   6   4  10   1         85.71              71.43        60.0   70.59   
2     N   6   0  15   0        100.00             100.00       100.0  100.00   

   Support  
0        8  
1        7  
2        6  

Confusion Matrix (rows=true, cols=pred):
        pred_Y  pred_F  pred_N
true_Y       4       4       0
true_F       1       6       0
true_N       0       0       6

=== Dev 指標（70 筆）===
Accuracy: 78.57%
Macro TPR(Recall): 79.07%
Macro TNR(Specificity): 89.11%

Per-class(metrics):
  class  TP  FP  TN  FN  TPR(recall)%  TNR(specificity)%  Precision%     F1%  \
0     Y  15   7  40   8         65.22              85.11       68.18   66.67   
1     F  18   8  37   7         72.00              82.22       69.2

## 迭代改 prompt 再次跑評估

In [5]:
train_dataset

[{'query_type': 'Y',
  'query': '這個帶花紋的茶壺感覺很溫柔，是不是和林黛玉在大觀園裡喝茶時的氛圍很搭？',
  'suppose_answer': 'Y',
  'predict_answer': 'F',
  'rejection_content': nan,
  'label': False},
 {'query_type': 'N',
  'query': '你更喜歡貓還是狗？',
  'suppose_answer': 'N',
  'predict_answer': 'N',
  'rejection_content': '您好，這個問題與《紅樓夢》展覽內容無關，歡迎您詢問有關展覽或小說的相關問題！',
  'label': True},
 {'query_type': 'Y',
  'query': '這個嵌著金色花紋的銀色小盒子是做什麼用的？',
  'suppose_answer': 'Y',
  'predict_answer': 'Y',
  'rejection_content': nan,
  'label': True},
 {'query_type': 'F',
  'query': '這一類的琺瑯壺，圖案那麼鮮明，應該很適合在賞花喝茶的園子用吧？',
  'suppose_answer': 'F',
  'predict_answer': 'F',
  'rejection_content': nan,
  'label': True},
 {'query_type': 'N',
  'query': '你喜歡夏天還是冬天？',
  'suppose_answer': 'N',
  'predict_answer': 'N',
  'rejection_content': '您好，這個問題與《紅樓夢》展覽內容無關，歡迎您詢問相關展品或小說內容！',
  'label': True},
 {'query_type': 'F',
  'query': '這種有鏤空紋路的指甲套，當時的貴族小姐應該也很講究保養外表吧？',
  'suppose_answer': 'F',
  'predict_answer': 'F',
  'rejection_content': nan,
  'label': True},

In [6]:
samples = [f"Query: {x['query']} Output: {x['label']}" for x in train_dataset]